# Plugin System: Extend the Pipeline Without Modifying Core Code

document-graph uses [pluggy](https://pluggy.readthedocs.io/) for extension points.
Plugins hook into the pipeline at defined stages without modifying core code:

```
Extract
    ↓
★ pre_transform hooks (audit trail, data quality, filtering)
    ↓
Transform
    ↓
★ post_transform hooks (enrichment, validation)
    ↓
★ validate_node hooks (per-node approval/rejection before write)
    ↓
Build (graph write)
    ↓
★ post_build hooks (metrics, alerts, lineage recording)
```

## Extension Points

| Hook | When | Use Case |
|------|------|----------|
| `pre_transform` | Before transformation | Add metadata, filter invalid records, audit stamp |
| `post_transform` | After transformation | Validate output, enrich with computed fields |
| `validate_node` | Before each node write | Reject nodes with bad IDs, blocked labels, missing props |
| `post_build` | After graph build | Collect metrics, send alerts, record lineage |

**Prerequisites:**
- `pip install graphrag-toolkit-document-graph` (pluggy is a core dependency)
- No Neptune or AWS credentials required

## 1. Create the Plugin Manager

In [ ]:
from graphrag_toolkit.document_graph.plugins import get_plugin_manager, hookimpl

pm = get_plugin_manager()
print(f'Plugin manager created: {pm.project_name}')
print(f'Registered hookspecs: {[name for name in dir(pm.hook) if not name.startswith("_")]}')

## 2. Register Sample Plugins

document-graph ships with sample plugins you can use as templates.

In [ ]:
from graphrag_toolkit.document_graph.plugins.samples import (
    AuditTrailPlugin,
    DataQualityPlugin,
    NodeValidatorPlugin,
    MetricsPlugin,
)

# Register plugins
audit = AuditTrailPlugin(pipeline_version='2.1.0')
quality = DataQualityPlugin(required_fields=['id', 'name'], max_properties=20)
validator = NodeValidatorPlugin(blocked_labels=['_Temp', '_Debug'])
metrics = MetricsPlugin()

pm.register(audit)
pm.register(quality)
pm.register(validator)
pm.register(metrics)

print(f'Registered {len(pm.get_plugins())} plugins:')
for p in pm.get_plugins():
    print(f'  - {type(p).__name__}')

## 3. Run pre_transform Hooks

Simulate the pipeline calling `pre_transform` — plugins process records in order.

In [ ]:
# Sample records — some valid, some missing required fields
raw_records = [
    {'id': 'u001', 'name': 'Alice', 'role': 'admin', 'email': 'alice@corp.com'},
    {'id': 'u002', 'name': 'Bob', 'role': 'analyst'},
    {'id': 'u003', 'name': '', 'role': 'viewer'},          # empty name — will be rejected
    {'id': '', 'name': 'Charlie', 'role': 'engineer'},     # empty id — will be rejected
    {'id': 'u005', 'name': 'Diana', 'role': 'lead'},
]

print(f'Input: {len(raw_records)} records')

# Call hooks — each plugin's pre_transform runs in order
results = pm.hook.pre_transform(records=raw_records)

# pluggy returns a list of results from all hooks
# The last result is from DataQualityPlugin (which filters)
# The first result is from AuditTrailPlugin (which stamps)
for hook_result in results:
    if hook_result is not None:
        processed = hook_result

print(f'Output: {len(processed)} records (after quality filter)')
print(f'Rejected: {len(quality.rejected)} records')
print()
print('Passed records:')
for r in processed:
    print(f'  {r["id"]}: {r["name"]} (stamped: {r.get("_pipeline_version", "?")})')
print()
print('Rejected records:')
for r in quality.rejected:
    print(f'  {r["reason"]}')

## 4. Run validate_node Hooks

Before writing each node, the pipeline calls `validate_node` — plugins can
reject nodes that don't meet their criteria.

In [ ]:
test_nodes = [
    {'node_id': 'u001', 'labels': ['User'], 'properties': {'name': 'Alice'}},
    {'node_id': 'u002', 'labels': ['User'], 'properties': {'name': 'Bob'}},
    {'node_id': '', 'labels': ['User'], 'properties': {'name': 'Ghost'}},           # empty ID
    {'node_id': 'tmp1', 'labels': ['_Temp'], 'properties': {'name': 'TempNode'}},   # blocked label
    {'node_id': 'u005', 'labels': ['User'], 'properties': {'name': 'Diana'}},
]

print('Node validation results:')
for node in test_nodes:
    results = pm.hook.validate_node(
        node_id=node['node_id'],
        labels=node['labels'],
        properties=node['properties']
    )
    # All hooks must return True for the node to pass
    approved = all(r for r in results if r is not None)
    status = '✅ WRITE' if approved else '❌ SKIP'
    print(f'  {status} {node["node_id"] or "(empty)":8s} {node["labels"]} — {node["properties"]["name"]}')

print(f'\nSkipped by validator: {len(validator.skipped)}')
for s in validator.skipped:
    print(f'  {s}')

## 5. Run post_build Hooks

After the graph build, plugins receive build statistics for monitoring.

In [ ]:
# Simulate build stats
build_stats = {
    'nodes_written': 3,
    'edges_written': 2,
    'duration_seconds': 1.5,
    'tenant': 'plugin_demo',
}

pm.hook.post_build(stats=build_stats)

print('Metrics plugin summary:')
summary = metrics.get_summary()
for k, v in summary.items():
    print(f'  {k}: {v}')

## 6. Write Your Own Plugin

Create a plugin by defining a class with `@hookimpl` methods:

In [ ]:
class MyCustomPlugin:
    """Example: add a computed 'full_label' field to every record."""

    @hookimpl
    def post_transform(self, records: list) -> list:
        for record in records:
            name = record.get('name', 'Unknown')
            role = record.get('role', 'none')
            record['full_label'] = f'{name} ({role})'
        return records

# Register and test
my_plugin = MyCustomPlugin()
pm.register(my_plugin)

test = [{'name': 'Alice', 'role': 'admin'}, {'name': 'Bob', 'role': 'analyst'}]
results = pm.hook.post_transform(records=test)

for hook_result in results:
    if hook_result:
        for r in hook_result:
            print(f'  {r}')

## Summary

The plugin system lets you extend document-graph without forking or modifying core code:

```python
# 1. Define your plugin
class MyPlugin:
    @hookimpl
    def pre_transform(self, records):
        # your logic here
        return records

# 2. Register it
pm = get_plugin_manager()
pm.register(MyPlugin())

# 3. Pipeline calls your hooks automatically
```

**Built-in sample plugins** (use as templates):
- `AuditTrailPlugin` — stamps records with timestamp + version
- `DataQualityPlugin` — rejects records missing required fields
- `NodeValidatorPlugin` — blocks nodes with empty IDs or forbidden labels
- `MetricsPlugin` — collects build statistics for monitoring